In [1]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader


class SequentialMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=(128, 64), output_dim=10, dropout=0.2):  # output_dim 기본값을 10으로 수정
        super().__init__() 
        h1, h2 = hidden_dim
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.ReLU(),

            nn.Dropout(p=dropout),

            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Dropout(p=dropout),

            nn.Linear(h2, output_dim)
        )

    def forward(self, x):
        return self.net(x)
    

digits = load_digits()
X_train, X_temp, y_train, y_temp = train_test_split(digits.data, digits.target, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

train_dataset = TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train))
val_dataset = TensorDataset(torch.FloatTensor(X_val), torch.LongTensor(y_val))
test_dataset = TensorDataset(torch.FloatTensor(X_test), torch.LongTensor(y_test))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

model = SequentialMLP(input_dim=64, hidden_dim=(128, 64), output_dim=10, dropout=0.2)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


import copy
# state_dict(): 모델의 파라미터(가중치와 편향)를 딕셔너리 형태로 반환
    # 모델의 파라미터를 쉽게 저장하고 불러오기 위해
# 조기 종료를 구현하려면 가장 성능이 좋았던 모델을
# 따로 저장해야 하기 때문에 state_dict()를 사용
# deepcopy(): 객체를 깊은 복사
    # 모델의 파라미터를 복사할 때 원본 모델에 영향을 주지 않도록 하기 위해 사용
best_model_wts = copy.deepcopy(model.state_dict())
best_val_loss = float('inf')
patience = 3
counter = 0

num_epochs = 50
for epoch in range(num_epochs):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

    # 위 까지는 기존의 학습 루프와 동일

    # 에폭이 끝날 때마다 검증 손실 계산
    # eval() 모드로 전환 (Dropout 같은 기능 비활성화)
    model.eval()
    val_loss = 0.0
    # no_grad() 컨텍스트 매니저: 역전파 과정에서 기울기 계산을 하지 않도록 함
        # 검증 단계에서는 모델의 파라미터를 업데이트하지 않기 때문에
        # 기울기 계산이 필요 없음
    with torch.no_grad():
        for data, target in val_loader:
            output = model(data)
            loss = criterion(output, target)
            val_loss += loss.item() * data.size(0)  # 배치 손실을 누적

    val_loss /= len(val_loader.dataset)  # 평균 검증 손실 계산
    print(f'Epoch {epoch+1}, Validation Loss: {val_loss:.4f}')

    # 검증 손실이 개선되었는지 확인
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        counter = 0  # 카운터 초기화
    else:
        counter += 1
        if counter >= patience:
            print("조기 종료: 검증 손실이 개선되지 않음")
            break

Epoch 1, Validation Loss: 1.4111
Epoch 2, Validation Loss: 0.4838
Epoch 3, Validation Loss: 0.2625
Epoch 4, Validation Loss: 0.1737
Epoch 5, Validation Loss: 0.1374
Epoch 6, Validation Loss: 0.1257
Epoch 7, Validation Loss: 0.1079
Epoch 8, Validation Loss: 0.0942
Epoch 9, Validation Loss: 0.0904
Epoch 10, Validation Loss: 0.0874
Epoch 11, Validation Loss: 0.0807
Epoch 12, Validation Loss: 0.0774
Epoch 13, Validation Loss: 0.0850
Epoch 14, Validation Loss: 0.0807
Epoch 15, Validation Loss: 0.0729
Epoch 16, Validation Loss: 0.0703
Epoch 17, Validation Loss: 0.0796
Epoch 18, Validation Loss: 0.0784
Epoch 19, Validation Loss: 0.0701
Epoch 20, Validation Loss: 0.0633
Epoch 21, Validation Loss: 0.0644
Epoch 22, Validation Loss: 0.0742
Epoch 23, Validation Loss: 0.0747
조기 종료: 검증 손실이 개선되지 않음


1. 이미지 데이터 준비
    - 그림판으로 0부터 9까지 숫자를 직접 그려서 이미지 파일로 저장
2. 이미지 불러오기 (PIL 라이브러리 활용)
3. 크기 조절 (이미지 훈련 데이터와 같은 8x8 픽셀로 스케일링)
4. 흑백 변환
5. 데이터 형식 변환 (1차원 벡터 - 64개 특성과 PyTorch 텐서로 변환)
6. 표준화

In [2]:
# 1. 이미지 데이터 준비

# 2. 이미지 불러오기 (PIL 라이브러리 활용)
from PIL import Image
import numpy as np
import torch

image = Image.open('ymn2.png')
print(f"원본 이미지 크기: {image.size}")

# 3. 크기 조절 (이미지 훈련 데이터와 같은 8x8 픽셀로 스케일링)
image = image.resize((8, 8))

# 4. 흑백 변환
image = image.convert('L')

# 5. 데이터 형식 변환 (1차원 벡터 - 64개 특성과 PyTorch 텐서로 변환)
image_array = np.array(image).reshape(-1)  # 1차원 벡터로 변환

# digits 데이터셋과 동일한 픽셀 값 범위로 변환 (0-255 -> 0-8)
image_array = image_array / 255.0 * 8.0

# 6. 표준화 (훈련 데이터와 동일한 StandardScaler 적용)
image_array = image_array.reshape(1, -1)  # (1, 64) 형태로 변환
image_normalized = scaler.transform(image_array)  # 훈련 때 사용한 scaler로 표준화
image_tensor = torch.FloatTensor(image_normalized)  # PyTorch 텐서로 변환

원본 이미지 크기: (8, 8)


1. **가중치 불러오기**
    - `model.load_state_dict(best_model_wts)`로 저장해둔 최적의 가중치를 불러옴
2. **평가 모드 전환**
    - `model.eval()`을 호출해서 모델을 '평가 모드'로 설정
3. **예측 실행**
    - 전처리된 이미지 텐서를 모델에 입력해서 예측값(logits)을 얻음 `logits = model(test_tensor)`
4. **결과 해석**
    - 모델이 출력한 `logits`는 각 숫자(0~9)에 대한 점수.
    - `torch.argmax()` 함수를 사용해서 가장 높은 점수를 받은 인덱스를 찾으면, 모델이 예측한 숫자

In [3]:
# 2. 최적의 모델 파라미터 로드
model.load_state_dict(best_model_wts)

# 3. 평가 모드 전환
model.eval()

# 4. 예측 실행
logits = model(image_tensor)

# 5. 결과 해석
predicted_class = logits.argmax(dim=1)
print(logits)
print(f"모델이 예측한 숫자: {predicted_class.item()}")

tensor([[-55.5838,  -1.0303,  -1.8869, -39.0751, -57.9295,   9.0021, -44.6797,
         -14.1846, -53.2233, -49.5765]], grad_fn=<AddmmBackward0>)
모델이 예측한 숫자: 5
